# Condition encoder（VA + 音楽条件スコア）

既存 `colab_train.ipynb` とは分離。自作VA教師は使わない。

データ: THETA / WRIME / Warriner mood語 / Jamendo 実タグ
出力: `checkpoints/va_encoder/jp_condition_*`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/vaEncoder"
REPO = "/content/kenkyuu"

!pip -q install transformers datasets sentencepiece fugashi unidic-lite


In [ ]:
# リポジトリ配置（初回）
# !git clone -b feat/condition-encoder https://github.com/Kakeru0307/kenkyuu.git {REPO}
import os, sys
os.makedirs(DRIVE_ROOT, exist_ok=True)
sys.path.insert(0, f"{REPO}/prttype")
print("ok", REPO)

## raw 自動取得（ブラウザDL不要）

次のセルで Warriner / Jamendo TSV を Drive の `vaEncoder/raw` に落とす。
WRIME は TSV が無ければ後段が HF 経由。THETA は別途 clone。

In [ ]:
import os
import urllib.request

RAW = f"{DRIVE_ROOT}/raw"
os.makedirs(RAW, exist_ok=True)

DOWNLOADS = {
    "Ratings_Warriner_et_al.csv": "https://raw.githubusercontent.com/JULIELab/XANEW/master/Ratings_Warriner_et_al.csv",
    "autotagging_moodtheme.tsv": "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/autotagging_moodtheme.tsv",
    "autotagging_genre.tsv": "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/autotagging_genre.tsv",
    "wrime-ver2.tsv": "https://raw.githubusercontent.com/ids-cv/wrime/master/wrime-ver2.tsv",
}

def fetch(name: str, url: str) -> None:
    dest = os.path.join(RAW, name)
    if os.path.isfile(dest) and os.path.getsize(dest) > 1000:
        print("exists", name, os.path.getsize(dest))
        return
    print("downloading", name, "...")
    urllib.request.urlretrieve(url, dest)
    print("ok", name, os.path.getsize(dest))

for name, url in DOWNLOADS.items():
    fetch(name, url)

# THETA（音楽キャプション＋人手VA）
THETA_DIR = f"{RAW}/sakai25_music_dataset"
if not os.path.isdir(THETA_DIR):
    !git clone --depth 1 https://github.com/takamichi-lab/sakai25_music_dataset.git {THETA_DIR}
else:
    print("exists THETA", THETA_DIR)

!ls -la {RAW}

In [ ]:
import os
DATA = f"{DRIVE_ROOT}/jsonl"
os.makedirs(DATA, exist_ok=True)
SCRIPTS = f"{REPO}/prttype/scripts"

# Drive 配置:
#   {DRIVE_ROOT}/raw/wrime-ver2.tsv
#   {DRIVE_ROOT}/raw/sakai25_music_dataset/   (THETA clone)
#   {DRIVE_ROOT}/raw/Ratings_Warriner_et_al.csv
#   {DRIVE_ROOT}/raw/autotagging_moodtheme.tsv
#   {DRIVE_ROOT}/raw/autotagging_genre.tsv
WRIME_TSV = f"{DRIVE_ROOT}/raw/wrime-ver2.tsv"
THETA_DIR = f"{DRIVE_ROOT}/raw/sakai25_music_dataset"
WARRINER = f"{DRIVE_ROOT}/raw/Ratings_Warriner_et_al.csv"
JAMENDO_MOOD = f"{DRIVE_ROOT}/raw/autotagging_moodtheme.tsv"
JAMENDO_GENRE = f"{DRIVE_ROOT}/raw/autotagging_genre.tsv"

# WRIME（TSV が無ければ HF）
if os.path.isfile(WRIME_TSV):
    !python {SCRIPTS}/build_wrime_va_pairs.py --csv {WRIME_TSV} --out {DATA}/wrime_va.jsonl
else:
    !python {SCRIPTS}/build_wrime_va_pairs.py --hf --out {DATA}/wrime_va.jsonl

# THETA
if os.path.isdir(THETA_DIR):
    !python {SCRIPTS}/build_theta_va_pairs.py --theta-dir {THETA_DIR} --out {DATA}/theta_va.jsonl
else:
    print("skip THETA (dir missing)")

# Warriner mood
if os.path.isfile(WARRINER):
    !python {SCRIPTS}/build_warriner_mood_va_pairs.py --warriner {WARRINER} --out {DATA}/warriner_mood_va.jsonl
else:
    print("skip Warriner")

# Jamendo axes（mood + genre）
jamendo = [p for p in (JAMENDO_MOOD, JAMENDO_GENRE) if os.path.isfile(p)]
if jamendo:
    tsv_args = " ".join(jamendo)
    !python {SCRIPTS}/build_music_condition_pairs.py --tsv {tsv_args} --out {DATA}/music_axes.jsonl --max-rows 20000
else:
    print("skip Jamendo")

!ls -la {DATA}

## 学習

In [ ]:
OUT = f"{DRIVE_ROOT}/ckpt/jp_condition_last"
jsonls = []
for name in ("wrime_va.jsonl", "theta_va.jsonl", "warriner_mood_va.jsonl", "music_axes.jsonl"):
    p = f"{DATA}/{name}"
    if os.path.isfile(p):
        jsonls.append(p)
assert jsonls, "JSONL が無い"
cmd = (
    f"python {SCRIPTS}/train_condition_encoder.py "
    + " ".join(f"--jsonl {p}" for p in jsonls)
    + f" --out {OUT} --epochs 3 --batch-size 16"
)
print(cmd)
!{cmd}

## スモーク（軸分離・中立A）

In [ ]:
import sys
sys.path.insert(0, f"{REPO}/prttype")
from condition_encoder import analyze_condition

prompts = [
    "普通の曲",
    "作業BGM",
    "激しくて熱い曲",
    "静かな曲",
    "rock music",
    "jazz music",
    "relaxing calm music",
]
for t in prompts:
    r = analyze_condition(t, ckpt_path=OUT)
    print(t, r["va"], {k: round(v, 3) for k, v in r["music_scores"].items() if v > 0.2})